# Lesson 2 — 범주형과 순환형 인코딩

**예상 시간:** 개념 40분 + 실습 70분  
**오늘의 새 산출물:** 범주형·순환형 표현의 증분 가치를 측정한 공통 backtest

Lesson 1 답안이 통과된 후 시작합니다. 실제 과제의 완성 코드는 포함하지 않습니다.

## 1. 오늘의 질문

> `weather=3`은 `weather=1`보다 정확히 세 배 나쁜 날씨일까? 23시와 0시는 숫자로는 멀지만 시간으로도 멀까?

데이터가 숫자로 저장되었다고 모두 크기와 거리를 가진 연속값은 아닙니다. 오늘은 값의 **의미에 맞는 표현**을 모델에 전달합니다.

## 2. 선수 지식 확인

- Lesson 1의 안전한 Feature Set A/B와 3개 fold를 재사용할 수 있는가?
- `fit`은 학습 규칙을 정하고 `transform`은 그 규칙을 적용한다는 차이를 말할 수 있는가?
- 새로운 feature의 유용성을 같은 validation 행에서 비교해야 한다는 점을 기억하는가?

## 3. 개념 설명

### 3.1 범주형 숫자

`weather`의 1, 2, 3, 4는 날씨 종류의 이름표입니다. 4가 2의 두 배라는 산술 의미가 없습니다. `season`, `weather`, `weekday`도 같은 성격입니다. One-hot encoding은 각 범주에 별도 스위치를 만들어 이 잘못된 거리 가정을 제거합니다.

`OneHotEncoder(handle_unknown='ignore')`는 train에서 범주 목록을 학습하고 새 데이터를 여러 0/1 열로 변환합니다. `.fit_transform()`은 학습과 변환을 함께 하고, `.transform()`은 이미 학습한 규칙만 적용합니다. 새 객체를 반환하며 원본 DataFrame을 바꾸지 않습니다.

### 3.2 순환형 시간

시계에서 23시와 0시는 한 시간 차이지만 정수로는 23 차이입니다. 각도를 원 위의 x/y 좌표로 바꾸면 경계가 사라집니다.

`hour_sin = sin(2π × hour / 24)`, `hour_cos = cos(2π × hour / 24)`

두 열을 함께 사용해야 원 위의 위치가 구분됩니다. sin 하나만 쓰면 서로 다른 시간이 같은 값을 가질 수 있습니다. 월에는 주기 12를 사용합니다.

### 3.3 `ColumnTransformer`와 `Pipeline`

`ColumnTransformer`는 범주형 열에는 one-hot, 나머지 열에는 다른 규칙을 적용합니다. `Pipeline`은 전처리와 모델을 한 객체로 묶습니다. fold train에서 pipeline을 `fit`하면 encoder와 모델 모두 train만 보게 되어 실수를 줄입니다.

Tree 모델은 숫자 경계를 스스로 나눌 수 있어 cyclic feature가 항상 개선되지는 않습니다. 만들 수 있다는 이유가 아니라 OOF RMSLE로 채택합니다.

## 4. 손으로 만드는 작은 표

### One-hot

| weather | clear | mist | rain |
|---:|---:|---:|---:|
| clear | 1 | 0 | 0 |
| rain | 0 | 0 | 1 |

### 순환형 hour

| hour | sin | cos | 위치 |
|---:|---:|---:|---|
| 0 | 0 | 1 | 원의 위쪽 |
| 6 | 1 | 0 | 오른쪽 |
| 12 | 0 | -1 | 아래쪽 |
| 18 | -1 | 0 | 왼쪽 |

23시의 좌표는 0시 좌표 근처에 놓입니다.

## 5. 실행 가능한 toy example

실제 과제와 다른 카페 주문 데이터로 encoder와 pipeline의 입력·출력을 확인합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

toy = pd.DataFrame({
    'shop_type': ['office', 'park', 'office', 'station'],
    'hour': [0, 6, 12, 23],
    'temperature': [8.0, 12.0, 18.0, 9.0],
    'orders': [3, 11, 15, 5],
})

toy_x = toy.drop(columns='orders').copy()
toy_y = toy['orders'].copy()
toy_x['hour_sin'] = np.sin(2 * np.pi * toy_x['hour'] / 24)
toy_x['hour_cos'] = np.cos(2 * np.pi * toy_x['hour'] / 24)

categorical = ['shop_type']
numeric = ['hour', 'temperature', 'hour_sin', 'hour_cos']

preprocessor = ColumnTransformer([
    ('category', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('number', 'passthrough', numeric),
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=20, random_state=42)),
])

pipeline.fit(toy_x, toy_y)
toy_pred = pipeline.predict(toy_x)
transformed = pipeline.named_steps['preprocess'].transform(toy_x)

print('원본 X shape:', toy_x.shape)
print('변환 결과 type/shape:', type(transformed), transformed.shape)
print('예측 type/shape:', type(toy_pred), toy_pred.shape)
print('원본 컬럼은 유지:', toy_x.columns.tolist())

`ColumnTransformer.transform`은 새 행렬을 반환하고 원본을 바꾸지 않습니다. `Pipeline.fit`은 pipeline 자신을 반환하며 내부 encoder와 모델 상태를 학습합니다.

잘못된 패턴은 전체 데이터에 encoder를 먼저 `fit_transform`한 뒤 fold를 나누는 것입니다. 범주 목록 같은 전처리 규칙도 fold train에서만 학습하도록 pipeline 전체를 fold 안에서 `fit`해야 합니다.

## 6. Data Leakage 점검

- OneHotEncoder가 각 fold의 train에서만 fit되는가?
- validation에 처음 등장한 범주를 `handle_unknown='ignore'`로 안전하게 처리하는가?
- train과 validation에 같은 feature 함수가 적용되는가?
- 실제 target 또는 test 예측 결과를 보고 범주를 합치지 않았는가?
- sin/cos 계산은 날짜 자체만 사용하므로 prediction time에 계산 가능한가?

## 7. 실제 데이터 Exercise

Lesson 1에서 더 나았던 Feature Set을 기준 B로 사용합니다. Feature Set C는 다음 표현을 추가하거나 교체합니다.

- 범주형: `season`, `weather`, `weekday`
- 이진형: `holiday`, `workingday`
- 연속형: `temp`, `atemp`, `humidity`, `windspeed`, `year`, `day`, `hour`
- 순환형: `hour_sin`, `hour_cos`, `month_sin`, `month_cos`

Feature Set B와 C는 Lesson 1과 같은 fold, 같은 RandomForest 설정, 같은 RMSLE/MAE로 비교합니다.

## 8. 코드 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/code/lesson2.ipynb`

- **C1 — 수작업 검증:** 실제 train에서 hour 0, 6, 12, 18, 23의 sin/cos 값을 한 행씩 출력해 예상 좌표와 비교한다.
- **C2 — Feature 함수:** 복사본을 입력받아 날짜 분해와 네 cyclic feature를 추가한 새 DataFrame을 반환한다. 입력 원본의 컬럼이 변하지 않았음을 출력한다.
- **C3 — Pipeline:** 지정 범주형·연속형 열에 `ColumnTransformer`를 적용하고 고정 RandomForest와 `Pipeline`으로 묶는다. 변환 전/후 shape를 한 fold train에서 출력한다.
- **C4 — Backtest:** 각 fold 안에서 새 pipeline을 만들고 fit한다. Feature Set B/C의 fold별 RMSLE/MAE를 저장한다.
- **C5 — OOF 비교:** B/C 전체 OOF 점수와 Lesson 1의 선택 결과까지 포함한 비교표를 출력한다.

**코드 최소 통과 기준**

- sin/cos가 정확한 주기 24와 12를 사용한다.
- sin과 cos를 쌍으로 사용한다.
- encoder와 model이 fold train에서만 fit된다.
- validation 미지 범주가 오류 없이 처리된다.
- B/C가 동일한 행과 지표로 비교된다.

## 9. 글 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/text/lesson2.txt`

- **T1:** `weather=1`과 `weather=3`을 연속 숫자로만 다룰 때 모델이 할 수 있는 잘못된 가정을 설명한다. **통과 기준:** 이름표와 수량을 구분한다.
- **T2:** 23시와 0시를 사용해 cyclic encoding 전후 거리를 설명한다. **통과 기준:** sin/cos 두 열이 필요한 이유를 포함한다.
- **T3:** Feature Set B/C의 OOF RMSLE/MAE로 C의 채택 여부를 판단한다. **통과 기준:** tree 모델에서 cyclic feature가 반드시 개선된다고 단정하지 않는다.
- **T4:** pipeline이 leakage 위험을 줄이는 이유를 설명한다. **통과 기준:** encoder와 model 모두 fold train에서 fit된다는 점을 적는다.

## 10. 성찰 질문

**제출 불필요:** 표현 방식이 이론적으로 더 자연스럽더라도 validation 성능이 좋아지지 않았다면 그 feature를 최종 모델에 남겨야 할까요?

## 11. 제출 전 자체 점검

- [ ] C1~C5와 T1~T4가 모두 있는가?
- [ ] feature 함수가 입력 원본을 변경하지 않는가?
- [ ] fold 밖에서 encoder를 fit하지 않았는가?
- [ ] 다른 모델이나 split을 추가해 비교 조건을 바꾸지 않았는가?
- [ ] 실제 OOF 수치로 feature 채택 여부를 판단했는가?